# Bootstrap secrets

Run by hand, once per environment. Creates the `finhive` Databricks secret scope and populates every secret the code references by name. Values are typed in interactively (via widgets or `getpass`) - never hardcoded in this notebook.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists
import getpass

dbutils.widgets.text("scope", "finhive")
scope = dbutils.widgets.get("scope")


REQUIRED_KEYS = [
    "fred_api_key",
]

w = WorkspaceClient()

In [ ]:
def sanitize(value: str) -> str:
    return value.strip().lstrip("\ufeff")


try:
    w.secrets.create_scope(scope)
except ResourceAlreadyExists:
    pass

for key in REQUIRED_KEYS:
    raw_value = dbutils.widgets.get(key) if key in dbutils.widgets.getAll() else ""
    if not raw_value:
        raw_value = getpass.getpass(f"{key}: ")

    value = sanitize(raw_value)

    w.secrets.put_secret(scope=scope, key=key, string_value=value)
